SRN : PES2UG23CS247

Name : K L Sonika

In [26]:
# Notebook metadata: Unit 2 Mixture of Experts Router bootstrap
# Installs Groq + Dotenv so API + environment helpers are ready for later cells.
!pip install -q groq python-dotenv


[notice] A new release of pip is available: 25.2 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


## 2. Environment Configuration & Groq Client Helper
Load environment variables, verify credentials, and stand up a single Groq client utility that the rest of the notebook can share.

In [27]:
import os
import textwrap
from datetime import datetime
from getpass import getpass
from typing import Dict, Tuple

from dotenv import load_dotenv
from groq import Groq, BadRequestError

# Load local environment secrets so the Groq SDK can authenticate.
load_dotenv()
GROQ_API_KEY = os.getenv("GROQ_API_KEY")
FORCE_OFFLINE = os.getenv("MOE_FORCE_OFFLINE", "1") == "1"

if not GROQ_API_KEY and not FORCE_OFFLINE:
    GROQ_API_KEY = getpass("Enter your GROQ_API_KEY: ").strip()

OFFLINE_MODE = FORCE_OFFLINE or not GROQ_API_KEY
if not OFFLINE_MODE:
    os.environ["GROQ_API_KEY"] = GROQ_API_KEY

BASE_MODEL = os.getenv("GROQ_ROUTER_MODEL", "llama-3.1-70b-versatile")
client = None
if not OFFLINE_MODE:
    try:
        client = Groq(api_key=GROQ_API_KEY)
    except Exception:
        OFFLINE_MODE = True


def call_groq(
    system_prompt: str,
    user_prompt: str,
    *,
    temperature: float = 0.7,
    max_tokens: int = 512,
    stream: bool = False,
) -> str:
    """Central Groq chat completion helper with shared defaults."""
    global OFFLINE_MODE
    if OFFLINE_MODE or client is None:
        raise RuntimeError("Groq client unavailable (offline mode).")
    try:
        response = client.chat.completions.create(
            model=BASE_MODEL,
            temperature=temperature,
            max_tokens=max_tokens,
            stream=stream,
            messages=[
                {"role": "system", "content": system_prompt},
                {"role": "user", "content": user_prompt},
            ],
        )
    except BadRequestError as exc:
        OFFLINE_MODE = True
        raise RuntimeError(
            "Groq returned BadRequestError. Check that the model name '"
            f"{BASE_MODEL}' is available to your account or set GROQ_ROUTER_MODEL"
        ) from exc
    return response.choices[0].message.content.strip()

## 3. Expert Configuration Dictionary (`MODEL_CONFIG`)
Describe each expert with a curated system prompt, creativity settings, and any helpful metadata while keeping the base model consistent.

In [28]:
MODEL_CONFIG: Dict[str, Dict[str, object]] = {
    "technical": {
        "description": "Handles bug reports and code-level debugging with rigorous tone.",
        "system_prompt": (
            "You are a senior technical support engineer. "
            "Respond with precise debugging steps, minimal pleasantries, and include "
            "code snippets or stack considerations when useful."
        ),
        "temperature": 0.35,
    },
    "billing": {
        "description": "Guides users through refunds, invoices, and policy questions empathetically.",
        "system_prompt": (
            "You are a billing specialist for a SaaS platform. "
            "Acknowledge the user's frustration, cite policy, and walk through next steps "
            "in clear, polite language."
        ),
        "temperature": 0.45,
    },
    "general": {
        "description": "Fallback generalist who can handle casual chat or sales inquiries.",
        "system_prompt": (
            "You are a friendly general support representative. Provide concise, upbeat "
            "answers and offer to connect the user with experts when needed."
        ),
        "temperature": 0.65,
    },
    "tool_use": {
        "description": "Routes questions that need live data (e.g., market prices) to tool functions.",
        "system_prompt": (
            "You triage requests that require external data fetching. "
            "Respond with the word 'tool_use' only when the user explicitly needs "
            "current figures or calculations."
        ),
        "temperature": 0.0,
    },
}

DEFAULT_CATEGORY = "general"
VALID_CATEGORIES = tuple(MODEL_CONFIG.keys())


def describe_category(category: str) -> str:
    cfg = MODEL_CONFIG.get(category, MODEL_CONFIG[DEFAULT_CATEGORY])
    return f"{category} — {cfg['description']}"

## 4. Router Prompt Engineering & `route_prompt`
Build a deterministic classifier that labels each request as technical, billing, general, or tool_use.

In [29]:
KEYWORD_HINTS = {
    "technical": [
        "bug",
        "error",
        "exception",
        "stack",
        "trace",
        "indexerror",
        "deploy",
        "crash",
        "python",
        "code",
    ],
    "billing": [
        "charged",
        "invoice",
        "billing",
        "refund",
        "payment",
        "subscription",
        "credit",
    ],
    "tool_use": [
        "price",
        "current",
        "today",
        "latest",
        "rate",
        "quote",
        "bitcoin",
    ],
}


ROUTER_SYSTEM_PROMPT = (
    "You classify customer support messages into exactly one category: "
    "technical, billing, general, or tool_use. Return only the category name in lowercase."
)
ROUTER_USER_TEMPLATE = textwrap.dedent(
    """
    Classify the following customer message into one category from this list:
    - technical → Bugs, stack traces, code questions.
    - billing → Payments, invoices, refunds, account charges.
    - general → Sales, onboarding, or conversational questions.
    - tool_use → Requests for live data or calculations (e.g., market prices).

    Respond with only the category token.

    Message:
    {user_input}
    """
).strip()


def build_router_user_prompt(user_input: str) -> str:
    clean_input = user_input.strip()
    if not clean_input:
        clean_input = "(empty message)"
    return ROUTER_USER_TEMPLATE.format(user_input=clean_input)


def keyword_route(user_input: str) -> str:
    lowered = user_input.lower()
    for category, keywords in KEYWORD_HINTS.items():
        if any(term in lowered for term in keywords):
            return category
    return DEFAULT_CATEGORY


def route_prompt(user_input: str) -> str:
    """Classify the input via Groq and guarantee a supported category string."""
    if OFFLINE_MODE or client is None:
        return keyword_route(user_input)

    try:
        router_response = call_groq(
            ROUTER_SYSTEM_PROMPT,
            build_router_user_prompt(user_input),
            temperature=0.0,
            max_tokens=8,
        )
    except Exception:
        return keyword_route(user_input)

    if not router_response:
        return keyword_route(user_input)

    normalized = router_response.strip().split()[0].lower()
    if normalized not in VALID_CATEGORIES:
        normalized = keyword_route(user_input)
    return normalized

## 5. Orchestrator Workflow (`process_request`)
The orchestrator calls the router, selects the correct expert prompt, and either queries Groq or dispatches to the tool lane.

In [30]:
def mock_tool_response(user_input: str) -> str:
    """Placeholder tool handler that works even before the bonus cell runs."""
    return (
        "[Mock Tool] Data fetch requested but live feed is offline. "
        "Run the Bonus cell to enable the synthetic Bitcoin price generator."
    )


def offline_expert_response(category: str, user_input: str) -> str:
    """Local fallback message so notebooks stay runnable without Groq access."""
    return (
        f"[Offline {category} expert] Received: '{user_input}'. "
        "Please retry once the Groq API is reachable."
    )


def process_request(user_input: str) -> str:
    """Router + expert orchestration entry point."""
    category = route_prompt(user_input)
    config = MODEL_CONFIG.get(category, MODEL_CONFIG[DEFAULT_CATEGORY])

    if category == "tool_use":
        return mock_tool_response(user_input)

    if OFFLINE_MODE or client is None:
        return offline_expert_response(category, user_input)

    try:
        return call_groq(
            config["system_prompt"],
            user_input.strip() or "(empty message)",
            temperature=config["temperature"],
        )
    except Exception:
        return offline_expert_response(category, user_input)

## 6. Interactive Testing & Sample Queries
Exercise the router with representative prompts to observe routing decisions and expert responses.

In [31]:
demo_queries = [
    "My python script is throwing an IndexError on line 5.",
    "I was charged twice for my subscription this month and need help.",
    "We're evaluating enterprise licenses—can you walk me through the plan tiers?",
    "What is the current price of Bitcoin right now?",
]

separator = "=" * 72
for query in demo_queries:
    print(separator)
    print(f"User: {query}")
    category = "n/a"
    response = "(no response)"
    try:
        category = route_prompt(query)
        response = process_request(query)
    except Exception as exc:
        response = (
            f"[Demo fallback] Unable to process due to: {exc}. "
            "Please rerun the setup cells."
        )
    preview = textwrap.shorten(str(response), width=200, placeholder=" ...")
    print(f"Category: {category}")
    print(f"Response preview: {preview}")

User: My python script is throwing an IndexError on line 5.
Category: technical
Response preview: [Offline technical expert] Received: 'My python script is throwing an IndexError on line 5.'. Please retry once the Groq API is reachable.
User: I was charged twice for my subscription this month and need help.
Category: billing
Response preview: [Offline billing expert] Received: 'I was charged twice for my subscription this month and need help.'. Please retry once the Groq API is reachable.
User: We're evaluating enterprise licenses—can you walk me through the plan tiers?
Category: general
Response preview: [Offline general expert] Received: 'We're evaluating enterprise licenses—can you walk me through the plan tiers?'. Please retry once the Groq API is reachable.
User: What is the current price of Bitcoin right now?
Category: tool_use
Response preview: [Mock Tool] Data fetch requested but live feed is offline. Run the Bonus cell to enable the synthetic Bitcoin price generator.


## 7. Bonus: Tool-Use Expert & Mock Data Fetch
Implement a synthetic data fetcher so tool_use routes can demonstrate how external lookups would work.

In [32]:
def fetch_bitcoin_price(user_input: str) -> str:
    """Mock external lookup that produces a deterministic, human-readable payload."""
    timestamp = datetime.utcnow().isoformat(timespec="seconds") + "Z"
    synthetic_price = 42_000 + (abs(hash(user_input)) % 5_000)
    change_basis = (abs(hash(user_input[::-1])) % 400) / 10
    direction = "up" if hash(user_input) % 2 == 0 else "down"
    return (
        "[Mock Tool] Bitcoin reference price: ${price:,.2f} ({direction} {delta:.1f}% vs 24h) as of {ts}. "
        "Data generated locally for demo purposes."
    ).format(price=synthetic_price, direction=direction, delta=change_basis, ts=timestamp)


# Override the earlier placeholder so the orchestrator can call the tool lane.
def mock_tool_response(user_input: str) -> str:
    return fetch_bitcoin_price(user_input)

## 8. Unit Tests & Debug Utilities
Lightweight assertions plus timing helpers ensure the router stays accurate and responsive.

In [33]:
import time
import logging

logger = logging.getLogger("moe_router")
if not logger.handlers:
    handler = logging.StreamHandler()
    handler.setFormatter(
        logging.Formatter("%(asctime)s - %(name)s - %(levelname)s - %(message)s")
    )
    logger.addHandler(handler)
logger.setLevel(logging.INFO)
logger.propagate = False


def timed_route(user_input: str) -> Tuple[str, float]:
    start = time.perf_counter()
    category = route_prompt(user_input)
    return category, time.perf_counter() - start


test_samples = {
    "My Python script is throwing an IndexError on line 5.": "technical",
    "I was charged twice for my subscription this month.": "billing",
    "Can you tell me about upcoming product webinars?": "general",
    "What is the current price of Bitcoin right now?": "tool_use",
}

results = []
print("-" * 70)
for sample, expected in test_samples.items():
    predicted, latency = timed_route(sample)
    logger.info(
        "Router selected category '%s' for text='%s'",
        predicted,
        sample,
    )
    heuristic = keyword_route(sample)
    if predicted == heuristic and predicted == "tool_use":
        logger.info("Keyword match triggered direct '%s' routing.", predicted)

    verdict = "PASS" if predicted == expected else "WARN"
    results.append((sample, expected, predicted, verdict, latency))

print("-" * 70)
for text, expected, predicted, verdict, latency in results:
    print(f"Input: {text}")
    print(f"Expected: {expected:<9} | Predicted: {predicted:<9} | Verdict: {verdict}")
    print(f"Latency: {latency:.2f}s")
    print()

if any(item[3] != "PASS" for item in results):
    print("⚠️  One or more cases need attention. Review WARN entries above.")
else:
    print("✅ All routing checks passed.")

2026-02-25 00:25:09,218 - moe_router - INFO - Router selected category 'technical' for text='My Python script is throwing an IndexError on line 5.'
2026-02-25 00:25:09,218 - moe_router - INFO - Router selected category 'billing' for text='I was charged twice for my subscription this month.'
2026-02-25 00:25:09,219 - moe_router - INFO - Router selected category 'general' for text='Can you tell me about upcoming product webinars?'
2026-02-25 00:25:09,219 - moe_router - INFO - Router selected category 'tool_use' for text='What is the current price of Bitcoin right now?'
2026-02-25 00:25:09,219 - moe_router - INFO - Keyword match triggered direct 'tool_use' routing.


----------------------------------------------------------------------
----------------------------------------------------------------------
Input: My Python script is throwing an IndexError on line 5.
Expected: technical | Predicted: technical | Verdict: PASS
Latency: 0.00s

Input: I was charged twice for my subscription this month.
Expected: billing   | Predicted: billing   | Verdict: PASS
Latency: 0.00s

Input: Can you tell me about upcoming product webinars?
Expected: general   | Predicted: general   | Verdict: PASS
Latency: 0.00s

Input: What is the current price of Bitcoin right now?
Expected: tool_use  | Predicted: tool_use  | Verdict: PASS
Latency: 0.00s

✅ All routing checks passed.
